In [ ]:
# =============================================================================
# 功能：根据预设规则过滤JSON数据中的QA（问答）对象。保留所有选择题（含"options"
#       字段）以及答案为纯数字（整数或小数）的简答题；删除答案为非数字的
#       简答题。处理过程中保留原有条目结构，并对过滤后的QA重新编号（QA1~QAn），
#       输出新的JSON文件，同时统计原始QA总数、删除数量及受影响的条目数，
#       用于数据清洗和精简数据集。
# =============================================================================
import json
import re

# ================== 配置 ==================
INPUT_JSON = "1.json"      # 输入的大 JSON 文件路径
OUTPUT_JSON = "filtered_qa.json" # 输出文件路径
# ==========================================

def is_pure_number(s):
    """判断字符串是否为纯数字（整数或浮点数，支持负号和小数点）"""
    if not isinstance(s, str):
        return False
    # 去除首尾空格
    s = s.strip()
    # 允许负数、小数点
    pattern = r'^-?\d+(?:\.\d+)?$'
    return bool(re.match(pattern, s))

def process_item(item_data, item_key):
    """处理单个条目的 QA，返回 (新的QA字典, 删除数量)"""
    qa_obj = item_data.get("QA")
    if not isinstance(qa_obj, dict):
        return {}, 0

    # 收集所有需要保留的 QA（按原顺序）
    kept_qa_list = []
    deleted_count = 0

    # 按 QA1 到 QA10 的顺序检查
    for i in range(1, 11):
        qa_name = f"QA{i}"
        qa = qa_obj.get(qa_name)
        if qa is None:
            continue

        # 判断是否为选择题（存在 options 字段）
        is_choice = "options" in qa and isinstance(qa.get("options"), list)
        if is_choice:
            # 选择题无条件保留
            kept_qa_list.append(qa)
        else:
            # 简答题：检查答案是否为纯数字
            answer = qa.get("answer", "")
            if is_pure_number(answer):
                kept_qa_list.append(qa)
            else:
                deleted_count += 1
                # 可选：打印调试信息
                # print(f"[{item_key}] 删除 {qa_name}，答案: {answer}")

    # 重新编号
    new_qa_obj = {}
    for idx, qa in enumerate(kept_qa_list, start=1):
        new_qa_obj[f"QA{idx}"] = qa

    return new_qa_obj, deleted_count

def main():
    with open(INPUT_JSON, 'r', encoding='utf-8') as f:
        full_data = json.load(f)

    new_data = {}
    total_deleted = 0
    total_original_qa = 0
    items_with_deletions = 0

    for item_key, item_data in full_data.items():
        # 统计原始 QA 数量（只统计存在的 QA1~QA10）
        qa_obj = item_data.get("QA", {})
        original_count = sum(1 for i in range(1, 11) if f"QA{i}" in qa_obj)
        total_original_qa += original_count

        new_qa, del_count = process_item(item_data, item_key)
        new_item = item_data.copy()
        new_item["QA"] = new_qa
        new_data[item_key] = new_item

        total_deleted += del_count
        if del_count > 0:
            items_with_deletions += 1

    # 保存新 JSON
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(new_data, f, indent=2, ensure_ascii=False)

    # 统计信息
    total_remaining = total_original_qa - total_deleted
    print("=" * 50)
    print("处理完成！")
    print(f"输入文件: {INPUT_JSON}")
    print(f"输出文件: {OUTPUT_JSON}")
    print("-" * 50)
    print(f"总条目数: {len(full_data)}")
    print(f"原始 QA 总数: {total_original_qa}")
    print(f"删除的简答题（非数字答案）数量: {total_deleted}")
    print(f"保留的 QA 数量: {total_remaining}")
    print(f"至少删除了一个 QA 的条目数: {items_with_deletions}")
    print("=" * 50)

if __name__ == "__main__":
    main()

In [ ]:
# =============================================================================
# 功能：统计JSON数据中所有QA（问答）对象的类型分布。根据QA是否包含"options"
#       字段区分为选择题和简答题；对于简答题，进一步判断答案内容是否为纯数字
#       （支持整数和小数，允许负号），分别统计数字型答案和非数字型答案的数量。
#       输出选择题总数、简答题总数、简答题中数字答案数及非数字答案数，用于
#       评估数据集的题目类型构成和答案格式分布。
# =============================================================================
import json
import re

# ================== 配置 ==================
JSON_FILE_PATH = ("data/qa/1.base/work__single_pdf__raw_mixed__n6204.json")   # 替换为你的大 JSON 文件路径
# ==========================================

def is_numeric_answer(answer_str):
    """判断答案字符串是否表示一个数字（整数或小数，允许负号）"""
    if not isinstance(answer_str, str):
        return False
    # 去除首尾空格
    s = answer_str.strip()
    # 匹配整数或小数（可选负号，数字，可选小数点和小数部分）
    pattern = r'^-?\d+(?:\.\d+)?$'
    return bool(re.match(pattern, s))

def main():
    with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    total_choice = 0      # 选择题总数
    total_short = 0       # 简答题总数
    short_numeric = 0     # 简答题中答案为纯数字的
    short_non_numeric = 0 # 简答题中答案为非纯数字的

    for item_key, item_data in data.items():
        qa_obj = item_data.get("QA")
        if not isinstance(qa_obj, dict):
            continue
        for qa_name, qa_content in qa_obj.items():
            # 只处理 QA1-QA10
            if not qa_name.startswith("QA"):
                continue
            # 判断是否有 options 字段
            if "options" in qa_content:
                total_choice += 1
            else:
                total_short += 1
                answer = qa_content.get("answer", "")
                if is_numeric_answer(answer):
                    short_numeric += 1
                else:
                    short_non_numeric += 1

    print("=== QA 统计结果 ===")
    print(f"选择题总数: {total_choice}")
    print(f"简答题总数: {total_short}")
    print(f"  - 简答题中答案为纯数字（包括小数点）: {short_numeric}")
    print(f"  - 简答题中答案为其他字符串: {short_non_numeric}")
    print(f"总计 QA 数量: {total_choice + total_short}")

if __name__ == "__main__":
    main()

In [ ]:
# =============================================================================
# 功能：验证JSON数据文件的结构完整性，根据预定义的规范检查每篇论文的元数据及
#       QA（问答）对象。检查内容包括：顶层字段（paper、primary_category、
#       secondary_category、QA）是否存在；二级分类是否属于对应一级学科的允许列表；
#       每个QA对象是否包含必填字段（question、answer、evidence_pages等）；
#       选项（options）结构是否完整；modal_types是否为允许的值（text/image/table/formula）；
#       question_type是否在Literal/Inferential范围内；question_category是否匹配
#       所属学科的允许类别列表。扫描整个文件后，汇总输出所有不符合规范的错误，
#       用于数据清洗和质量控制。
# =============================================================================
import json

# ================== 配置 ==================
JSON_FILE_PATH = "data/qa/1.base/work__single_pdf__raw_mixed__n6204.json"   # 替换为你的大 JSON 文件路径（历史过程文件；正式集合以当前 schema 为准）

# 允许的 modal_types 值
ALLOWED_MODAL_TYPES = {"text", "image", "table", "formula"}

# 允许的 question_type 值
ALLOWED_QUESTION_TYPES = {"Literal", "Inferential"}

# 学科 -> 允许的 question_category 描述（已去除前缀）
PRIMARY_TO_ALLOWED_CATEGORIES = {
    "Computer Science": {
        "Algorithm & Architecture Detail",
        "Experiment & Result Validation",
        "Method Innovation"
    },
    "Economics": {
        "Theoretical Framework & Concept Definition",
        "Empirical Design & Econometric Method",
        "Causal Inference & Result Interpretation"
    },
    "Electrical Engineering and Systems Science": {
        "System Architecture & Signal Processing Method",
        "Experiment & Performance Test",
        "System Optimization & Engineering Implementation"
    },
    "Mathematics": {
        "Definition & Theorem Statement",
        "Formula & Derivation Detail",
        "Proposition Proof & Logical Reasoning"
    },
    "Physics": {
        "Physical Concept & Theoretical Model",
        "Computation & Experimental Validation"
    },
    "Quantitative Biology": {
        "Biological Entity & Quantitative Model Definition",
        "Experimental Data & Statistical Analysis",
        "Biological Network & Dynamic Modeling"
    },
    "Quantitative Finance": {
        "Financial Theory & Pricing Model Definition",
        "Quantitative Strategy & Performance Analysis"
    },
    "Statistics": {
        "Statistical Concept & Probability Model Definition",
        "Statistical Inference & Method Application"
    }
}

# 学科 -> 允许的 secondary_category 列表（根据用户提供的映射）
PRIMARY_TO_ALLOWED_SECONDARY = {
    "Computer Science": [
        "Artificial Intelligence", "Computational Complexity", "Computers and Society",
        "Databases", "Data Structures and Algorithms", "Distributed Systems",
        "Neural and Evolutionary Computing", "Computer Science and Game Theory",
        "Computational Geometry", "Hardware Architecture", "Information Theory",
        "Multimedia", "Computation and Language", "Machine Learning",
        "Programming Languages", "Computational Engineering, Finance, and Science",
        "Cryptography and Security", "Social and Information Networks",
        "Computer Vision and Pattern Recognition"
    ],
    "Economics": [
        "Econometrics", "Theoretical Economics"
    ],
    "Electrical Engineering and Systems Science": [
        "Audio and Speech Processing", "Image and Video Processing", "Signal Processing"
    ],
    "Mathematics": [
        "Algebraic Geometry", "Analysis of PDEs", "Applied Probability", "Number Theory",
        "General Mathematics", "Geometry", "Category Theory", "Algebras",
        "Representation Theory", "Algebraic Topology"
    ],
    "Physics": [
        "Applied Plasma Physics", "Instrumentation and Methods for Astrophysics",
        "Atomic and Molecular Clusters", "Classical Physics", "Statistical Mechanics",
        "Cosmology and Nongalactic Astrophysics", "Soft Condensed Matter",
        "Space Biophysics", "High Energy Astrophysical Phenomena", "Mathematical Physics",
        "Nuclear Physics", "High Energy Physics", "Astrophysics of Galaxies",
        "Mesoscale and Nanoscale Physics", "Quantum Physics", "General Relativity and Quantum Cosmology"
    ],
    "Quantitative Biology": [
        "Biomolecules", "Cell Behavior", "Other Quantitative Biology", "Molecular Networks"
    ],
    "Quantitative Finance": [
        "Computational Finance", "Statistical Finance", "Mathematical Finance",
        "Portfolio and Risk Management"
    ],
    "Statistics": [
        "Applications and Methodology", "Computational Statistics", "Other Statistics"
    ]
}
# ==========================================

def check_qa_details(qa_data, qa_name, primary_cat, item_key):
    """检查单个 QA 中的 modal_types 和 question_category 和 question_type"""
    errors = []

    # 1. 检查 modal_types
    if "modal_types" in qa_data:
        modal_types = qa_data["modal_types"]
        if not isinstance(modal_types, list):
            errors.append(f"[{item_key}] -> {qa_name} 的 'modal_types' 不是数组")
        else:
            for mt in modal_types:
                if mt not in ALLOWED_MODAL_TYPES:
                    errors.append(f"[{item_key}] -> {qa_name} 的 modal_types 包含不允许的值 '{mt}'（允许: {ALLOWED_MODAL_TYPES}）")
    else:
        errors.append(f"[{item_key}] -> {qa_name} 缺少 'modal_types' 字段（可选）")

    # 2. 检查 question_type 取值
    if "question_type" in qa_data:
        qt = qa_data["question_type"]
        if not isinstance(qt, str):
            errors.append(f"[{item_key}] -> {qa_name} 的 'question_type' 不是字符串")
        elif qt not in ALLOWED_QUESTION_TYPES:
            errors.append(f"[{item_key}] -> {qa_name} 的 question_type '{qt}' 不在允许值 {ALLOWED_QUESTION_TYPES} 中")
    else:
        errors.append(f"[{item_key}] -> {qa_name} 缺少 'question_type' 字段")

    # 3. 检查 question_category
    if "question_category" not in qa_data:
        errors.append(f"[{item_key}] -> {qa_name} 缺少 'question_category' 字段")
        return errors

    q_cat = qa_data["question_category"]
    if not isinstance(q_cat, str):
        errors.append(f"[{item_key}] -> {qa_name} 的 'question_category' 不是字符串")
        return errors

    if primary_cat not in PRIMARY_TO_ALLOWED_CATEGORIES:
        errors.append(f"[{item_key}] 的 primary_category '{primary_cat}' 不在已知学科列表中，无法验证 question_category")
    else:
        allowed_set = PRIMARY_TO_ALLOWED_CATEGORIES[primary_cat]
        if q_cat not in allowed_set:
            errors.append(f"[{item_key}] -> {qa_name} 的 question_category '{q_cat}' 不属于学科 '{primary_cat}' 的允许子类 {allowed_set}")

    return errors

def main():
    with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
        full_data = json.load(f)

    all_errors = []

    for item_key, item_data in full_data.items():
        # 1. 检查顶层必要字段
        if "paper" not in item_data:
            all_errors.append(f"[{item_key}] 缺少顶层键 'paper'")
        if "primary_category" not in item_data:
            all_errors.append(f"[{item_key}] 缺少顶层键 'primary_category'")
        if "secondary_category" not in item_data:
            all_errors.append(f"[{item_key}] 缺少顶层键 'secondary_category'")
        if "QA" not in item_data:
            all_errors.append(f"[{item_key}] 缺少顶层键 'QA'")

        primary_cat = item_data.get("primary_category")
        secondary_cat = item_data.get("secondary_category")

        # 2. 检查 secondary_category 是否属于 primary_category 的允许列表
        if primary_cat and secondary_cat:
            if primary_cat not in PRIMARY_TO_ALLOWED_SECONDARY:
                all_errors.append(f"[{item_key}] primary_category '{primary_cat}' 不在已知学科列表中，无法验证 secondary_category")
            else:
                allowed_secondary = PRIMARY_TO_ALLOWED_SECONDARY[primary_cat]
                if secondary_cat not in allowed_secondary:
                    all_errors.append(f"[{item_key}] secondary_category '{secondary_cat}' 不属于 primary_category '{primary_cat}' 的允许列表 {allowed_secondary}")

        # 3. 检查 QA 对象
        qa_obj = item_data.get("QA")
        if not isinstance(qa_obj, dict):
            all_errors.append(f"[{item_key}] 'QA' 不是对象")
            continue

        # 4. 检查 QA1~QA10
        for qa_num in range(1, 11):
            qa_name = f"QA{qa_num}"
            if qa_name not in qa_obj:
                all_errors.append(f"[{item_key}] QA 中缺少 '{qa_name}'")
            else:
                qa_data = qa_obj[qa_name]
                if not isinstance(qa_data, dict):
                    all_errors.append(f"[{item_key}] {qa_name} 不是对象")
                else:
                    # 检查 QA 内部必要字段
                    for field in ["question", "answer", "evidence_pages", "question_type", "question_category"]:
                        if field not in qa_data:
                            all_errors.append(f"[{item_key}] -> {qa_name} 缺少字段 '{field}'")
                    # 如果有 options，检查其结构
                    if "options" in qa_data:
                        opts = qa_data["options"]
                        if not isinstance(opts, list):
                            all_errors.append(f"[{item_key}] -> {qa_name} 的 options 不是数组")
                        else:
                            for i, opt in enumerate(opts):
                                if not isinstance(opt, dict):
                                    all_errors.append(f"[{item_key}] -> {qa_name} 的 options[{i}] 不是对象")
                                else:
                                    if "id" not in opt:
                                        all_errors.append(f"[{item_key}] -> {qa_name} 的 options[{i}] 缺少 'id'")
                                    if "text" not in opt:
                                        all_errors.append(f"[{item_key}] -> {qa_name} 的 options[{i}] 缺少 'text'")
                    # 检查 evidence_pages 是否为整数列表
                    if "evidence_pages" in qa_data:
                        ep = qa_data["evidence_pages"]
                        if not isinstance(ep, list):
                            all_errors.append(f"[{item_key}] -> {qa_name} 的 evidence_pages 不是数组")
                        else:
                            for page in ep:
                                if not isinstance(page, int):
                                    all_errors.append(f"[{item_key}] -> {qa_name} 的 evidence_pages 包含非整数元素 {page}")
                    # 检查 question_type 和 question_category 的类型（类型检查在 check_qa_details 中也会做，但这里保留）
                    if "question_type" in qa_data and not isinstance(qa_data["question_type"], str):
                        all_errors.append(f"[{item_key}] -> {qa_name} 的 question_type 不是字符串")
                    if "question_category" in qa_data and not isinstance(qa_data["question_category"], str):
                        all_errors.append(f"[{item_key}] -> {qa_name} 的 question_category 不是字符串")

                    # 5. 检查 modal_types, question_type 值, question_category 学科归属
                    if primary_cat:
                        all_errors.extend(check_qa_details(qa_data, qa_name, primary_cat, item_key))

    # 输出结果
    if all_errors:
        print("发现以下结构错误：\n")
        for err in all_errors:
            print(err)
        print(f"\n共发现 {len(all_errors)} 个错误。")
    else:
        print("所有条目的结构均符合规范！")

if __name__ == "__main__":
    main()